# 🎥 Movie Recommender System with PySpark and Visualizations

### 📦 Load and explore data

In [1]:
from os import environ
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, explode, split, avg, countDistinct, desc

environ["SPARK_HADOOP_VERSION"] = "3.2.0"
spark = SparkSession.builder.appName("MovieRecommender").getOrCreate()


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).

25/04/22 16:22:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Py4JJavaError: An error occurred while calling None.org.apache.spark.api.java.JavaSparkContext.
: java.lang.UnsupportedOperationException: getSubject is not supported
	at java.base/javax.security.auth.Subject.getSubject(Subject.java:277)
	at org.apache.hadoop.security.UserGroupInformation.getCurrentUser(UserGroupInformation.java:577)
	at org.apache.spark.util.Utils$.$anonfun$getCurrentUserName$1(Utils.scala:2416)
	at scala.Option.getOrElse(Option.scala:189)
	at org.apache.spark.util.Utils$.getCurrentUserName(Utils.scala:2416)
	at org.apache.spark.SparkContext.<init>(SparkContext.scala:329)
	at org.apache.spark.api.java.JavaSparkContext.<init>(JavaSparkContext.scala:58)
	at java.base/jdk.internal.reflect.DirectConstructorHandleAccessor.newInstance(DirectConstructorHandleAccessor.java:62)
	at java.base/java.lang.reflect.Constructor.newInstanceWithCaller(Constructor.java:499)
	at java.base/java.lang.reflect.Constructor.newInstance(Constructor.java:483)
	at py4j.reflection.MethodInvoker.in

In [2]:
ratings_path = "ratings.csv"
movies_path = "movies.csv"

ratings_df = spark.read.option("header", True).option("inferSchema", True).csv(ratings_path)
movies_df = spark.read.option("header", True).option("inferSchema", True).csv(movies_path)


NameError: name 'spark' is not defined

In [ ]:
ratings_df.show(10)

In [ ]:
movies_df.show(10)

### 🧹 Data cleaning

In [ ]:
ratings_df = ratings_df.dropna().dropDuplicates()
movies_df = movies_df.dropna().dropDuplicates()


### 📊 General trends analysis

In [ ]:
movie_rating_stats = (
    ratings_df.groupBy("movieId")
    .agg(avg("rating").alias("avg_rating"), countDistinct("userId").alias("num_ratings"))
)

top_rated_movies = (
    movie_rating_stats.filter(col("num_ratings") >= 100)
    .join(movies_df, "movieId")
    .orderBy(desc("avg_rating"))
    .select("title", "avg_rating", "num_ratings")
)

top_rated_movies.show(10, truncate=False)


### 📈 Visualizations with seaborn

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

top_movies_pd = top_rated_movies.toPandas()

plt.figure(figsize=(10, 6))
sns.barplot(data=top_movies_pd.head(10), x="avg_rating", y="title", palette="viridis")
plt.title("Top 10 Highest Rated Movies (Min 100 ratings)")
plt.xlabel("Average Rating")
plt.ylabel("Movie Title")
plt.show()


In [ ]:
movies_with_genres = movies_df.withColumn("genre", explode(split(col("genres"), "\|")))

popular_genres = (
    movies_with_genres.groupBy("genre")
    .count()
    .orderBy(desc("count"))
)

popular_genres.show()


In [ ]:
popular_genres_pd = popular_genres.toPandas()

plt.figure(figsize=(12, 6))
sns.barplot(data=popular_genres_pd, x="count", y="genre", palette="cubehelix")
plt.title("Most Popular Genres")
plt.xlabel("Number of Movies")
plt.ylabel("Genre")
plt.show()


### 🧠 ALS Model Training

In [ ]:
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

(training_df, test_df) = ratings_df.randomSplit([0.8, 0.2], seed=42)

als = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    nonnegative=True,
    coldStartStrategy="drop"
)

model = als.fit(training_df)
predictions = model.transform(test_df)

evaluator = RegressionEvaluator(metricName="rmse", labelCol="rating", predictionCol="prediction")
rmse = evaluator.evaluate(predictions)
print(f"RMSE = {rmse:.4f}")


### 🧬 Content-Based Recommendation (TF-IDF on genres)

In [ ]:
from pyspark.ml.feature import CountVectorizer, IDF
from pyspark.sql.functions import split

movies_genre_df = movies_df.withColumn("genres_list", split(col("genres"), "\|"))

cv = CountVectorizer(inputCol="genres_list", outputCol="raw_features")
cv_model = cv.fit(movies_genre_df)
featurized_data = cv_model.transform(movies_genre_df)

idf = IDF(inputCol="raw_features", outputCol="features")
idf_model = idf.fit(featurized_data)
tfidf_data = idf_model.transform(featurized_data)


In [ ]:
from pyspark.ml.linalg import Vectors, Vector
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType

def cosine_similarity(v1: Vector, v2: Vector) -> float:
    dot = float(v1.dot(v2))
    norm1 = float(v1.norm(2))
    norm2 = float(v2.norm(2))
    return dot / (norm1 * norm2) if norm1 != 0 and norm2 != 0 else 0.0

cosine_similarity_udf = udf(cosine_similarity, DoubleType())

toy_story_vector = tfidf_data.filter(col("title") == "Toy Story (1995)").select("features").first()["features"]

similar_movies = tfidf_data.withColumn("similarity", cosine_similarity_udf(col("features"), Vectors.dense(toy_story_vector.toArray())))
similar_movies.orderBy(desc("similarity")).select("title", "genres", "similarity").show(10, truncate=False)


### 🧑‍🤝‍🧑 User-based Recommendation (KNN)

In [ ]:
user_movie_matrix = ratings_df.groupBy("userId", "movieId").agg(avg("rating").alias("rating"))
movie_ids = ratings_df.select("movieId").distinct().orderBy("movieId").rdd.map(lambda r: r[0]).collect()
movie_id_to_index = {movie_id: idx for idx, movie_id in enumerate(movie_ids)}

def build_user_vector(user_ratings):
    vec = [0.0] * len(movie_ids)
    for movie_id, rating in user_ratings:
        idx = movie_id_to_index[movie_id]
        vec[idx] = rating
    return Vectors.dense(vec)

from pyspark.sql import Row
user_vectors = (
    user_movie_matrix.rdd
    .map(lambda row: (row["userId"], (row["movieId"], row["rating"])))
    .groupByKey()
    .mapValues(list)
    .mapValues(build_user_vector)
    .map(lambda x: Row(userId=x[0], features=x[1]))
)

user_vector_df = spark.createDataFrame(user_vectors)

target_user_id = 1
target_vector = user_vector_df.filter(col("userId") == target_user_id).select("features").first()["features"]
user_vector_df = user_vector_df.withColumn("similarity", cosine_similarity_udf(col("features"), Vectors.dense(target_vector.toArray())))

top_k_neighbors = user_vector_df.filter(col("userId") != target_user_id).orderBy(desc("similarity")).limit(5)
top_k_neighbors_ids = [row["userId"] for row in top_k_neighbors.collect()]

neighbor_ratings = ratings_df.filter(col("userId").isin(top_k_neighbors_ids))
recommended_movies = (
    neighbor_ratings.join(ratings_df.filter(col("userId") == target_user_id), "movieId", "left_anti")
    .groupBy("movieId")
    .agg(avg("rating").alias("avg_rating"))
    .join(movies_df, "movieId")
    .orderBy(desc("avg_rating"))
)

recommended_movies.select("title", "avg_rating").show(10, truncate=False)


### 📊 Evaluation and Recommendations

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

def precision_at_k(recommendations_df, actual_df, k=10):
    window = Window.partitionBy("userId").orderBy(desc("prediction"))
    top_k_pred = recommendations_df.withColumn("rank", row_number().over(window)).filter(col("rank") <= k)
    joined = top_k_pred.join(actual_df, on=["userId", "movieId"], how="inner")
    precision = joined.groupBy("userId").agg((countDistinct("movieId") / lit(k)).alias("precision"))
    return precision.agg(avg("precision")).first()[0]

def coverage(recommendations_df, total_movies_count):
    recommended_movie_ids = recommendations_df.select("movieId").distinct().count()
    return recommended_movie_ids / total_movies_count

p_at_10 = precision_at_k(predictions, test_df, k=10)
movie_count = movies_df.select("movieId").distinct().count()
cov = coverage(predictions, movie_count)

print(f"Precision@10: {p_at_10:.4f}")
print(f"Coverage: {cov:.4f}")


In [ ]:
from pyspark.sql.functions import rand

fictitious_users = ratings_df.select("userId").distinct().orderBy(rand()).limit(3)
fictitious_ids = [row["userId"] for row in fictitious_users.collect()]

for uid in fictitious_ids:
    user_recommendations = model.recommendForAllUsers(10).filter(col("userId") == uid)
    exploded = user_recommendations.selectExpr("userId", "explode(recommendations) as rec")                                    .select("userId", col("rec.movieId").alias("movieId"), col("rec.rating").alias("score"))                                    .join(movies_df, "movieId")                                    .select("title", "score")                                    .orderBy(desc("score"))

    print(f"\nTop recommendations for user {uid}:")
    exploded.show(10, truncate=False)


### 🧾 Conclusion

- **ALS** provides strong collaborative filtering performance.
- **TF-IDF based filtering** helps find content-similar movies, especially useful in cold-start scenarios.
- **KNN** offers personalized recommendations by leveraging similar users.
- **Best strategy**: use ALS for scale, and mix with content-based for richer personalization.
